# PhoBERT trên ViCTSD

Fine-tune PhoBERT hai lần: một lần với nhãn gốc, một lần với nhãn AI re-annotated.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/lechihoang/multi-agent-annotation.git'
PROJECT_DIR = Path('/kaggle/working/multi-agent-annotation')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR / 'notebooks')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR), 'torch', 'transformers', 'datasets', 'accelerate'], check=True)

sys.path.insert(0, str(PROJECT_DIR))
root = PROJECT_DIR
print(f'Project root: {PROJECT_DIR}')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

from src.experiment_utils import (
    compare_prediction_disagreements,
    compare_predictions,
    evaluate_predictions,
    load_victsd_splits,
    normalize_text,
)

MODEL_NAME = 'PhoBERT'
BASE_MODEL = 'vinai/phobert-base-v2'
MAX_LEN = 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
splits = load_victsd_splits(root / 'data')
y_test = splits['test']['Constructiveness'].astype(int).values

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_hf_dataset(df):
    return Dataset.from_dict({
        'text': df['Comment'].map(normalize_text).tolist(),
        'label': df['Constructiveness'].astype(int).tolist(),
    })

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LEN)

train_dataset_orig = to_hf_dataset(splits['train_orig']).map(tokenize, batched=True)
train_dataset_new = to_hf_dataset(splits['train_new']).map(tokenize, batched=True)
test_dataset = to_hf_dataset(splits['test']).map(tokenize, batched=True)

In [ ]:
def train_and_predict(train_dataset, eval_dataset, run_name):
    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
    args = TrainingArguments(
        output_dir=f'./phobert_results_{run_name}',
        eval_strategy='epoch',
        save_strategy='no',
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to='none',
        seed=42,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=eval_dataset)
    trainer.train()
    output = trainer.predict(eval_dataset)
    return np.argmax(output.predictions, axis=-1)

print('Training PhoBERT on original labels...')
y_pred_orig = train_and_predict(train_dataset_orig, test_dataset, 'original')
evaluate_predictions('PhoBERT (Original labels)', y_test, y_pred_orig)

print('Training PhoBERT on AI re-annotated labels...')
y_pred_new = train_and_predict(train_dataset_new, test_dataset, 'reannotated')
evaluate_predictions('PhoBERT (AI re-annotated labels)', y_test, y_pred_new)

comparison = compare_predictions(MODEL_NAME, y_test, y_pred_orig, y_pred_new)

In [ ]:
diff_df = compare_prediction_disagreements(
    splits['test']['Comment'].map(normalize_text),
    y_test,
    y_pred_orig,
    y_pred_new,
)
diff_df.head(10)